
# Simulação da Equação Swing para um Gerador Síncrono

Este notebook simula o comportamento dinâmico de um **gerador síncrono** usando a **equação swing**, que modela o movimento do rotor em relação ao referencial síncrono.

A equação swing descreve como o ângulo do rotor \( \delta(t) \) e sua velocidade angular \( \omega(t) \) evoluem no tempo, considerando a potência mecânica fornecida pela turbina e a potência elétrica entregue à rede.

## Equação swing (forma clássica):

\[
M \frac{d^2\delta}{dt^2} + D \frac{d\delta}{dt} = P_m - P_e
\]

Onde:
- \( M \): constante de inércia (MW·s/el.rad)
- \( D \): coeficiente de amortecimento (pu)
- \( \delta(t) \): ângulo do rotor (rad)
- \( \omega(t) = \dot{\delta}(t) \): velocidade angular (rad/s)
- \( P_m \): potência mecânica de entrada (pu)
- \( P_e = \frac{EV}{X} \sin(\delta) \): potência elétrica entregue (pu)

A equação é não linear devido ao termo \( \sin(\delta) \), e é fundamental em estudos de **estabilidade transitória** de sistemas elétricos de potência.

Nesta simulação, usamos valores típicos para os parâmetros e condições iniciais com \( \delta(0) = 30^\circ \) e \( \omega(0) = 0 \).


In [7]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

# Parâmetros do sistema
M = 5.0
D = 1.0
P_m = 1.0
E = 1.1
V = 1.0
X = 0.6

# Equação swing
def swing_equation(t, y):
    delta = y[0]
    omega = y[1]
    ddelta_dt = omega
    domega_dt = (P_m - (E * V / X) * np.sin(delta) - D * omega) / M
    return [ddelta_dt, domega_dt]

# Condições iniciais
y0 = [np.deg2rad(30), 0.0]
t_span = (0, 50)
t_eval = np.linspace(*t_span, 50000)

# Solução
sol = solve_ivp(swing_equation, t_span, y0, t_eval=t_eval)
t = sol.t
delta = np.rad2deg(sol.y[0])
omega = sol.y[1]
P_e = (E * V / X) * np.sin(np.deg2rad(delta))

# Figura com 3 eixos Y
fig = go.Figure()

# Eixo da esquerda: Ângulo δ(t)
fig.add_trace(go.Scatter(
    x=t, y=delta, name="Ângulo δ(t) [°]",
    line=dict(color='blue'),
    yaxis="y"
))

# Eixo da direita (interno): Velocidade angular ω(t)
fig.add_trace(go.Scatter(
    x=t, y=omega, name="Velocidade angular ω(t) [rad/s]",
    line=dict(color='green', dash='dash'),
    yaxis="y2"
))

# Eixo da direita (externo): Potência elétrica P_e(t)
fig.add_trace(go.Scatter(
    x=t, y=P_e, name="Potência elétrica P_e(t) [pu]",
    line=dict(color='red', dash='dot'),
    yaxis="y3"
))

# Layout com três eixos Y
fig.update_layout(
    title="Equação Swing: δ(t), ω(t) e Pₑ(t)",
    xaxis=dict(title="Tempo [s]"),

    yaxis=dict(  # Eixo para δ(t)
        title="Ângulo δ(t) [°]",
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue"),
        position=0.0
    ),

    yaxis2=dict(  # Eixo para ω(t)
        title="Velocidade angular ω(t) [rad/s]",
        titlefont=dict(color="green"),
        tickfont=dict(color="green"),
        overlaying="y",
        side="right",
        position=0.93
    ),

    yaxis3=dict(  # Eixo para Pₑ(t)
        title="Potência elétrica Pₑ(t) [pu]",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        anchor="free",
        position=1.0
    ),

    legend=dict(x=0.01, y=0.99),
    width=2000,
    height=500
)

fig.show()


Output hidden; open in https://colab.research.google.com to view.